## Project 1 Cross-Reference - in respect of Day 6 Findings

All Q3 findings from Project 1 SQL analysis (Days 17-22) verified in pandas:
- IQR outlier thresholds: matched to within 3% (windowing difference)
- Outlier concentration: confirmed and *sharpened* , mega-cap dominance is 
  exclusive, not just dominant
- Tech sector tailwind: confirmed; margin story revised with more nuance
- Electronics divergence: matched precisely (INTC 4.6%, NVDA 17.2%, 35.8× 
  market cap ratio)

The pandas replication has revealed more precise analytical framing in 
three cases (Twin NPM Peaks composition, outlier exclusivity, Tech margin 
stability). This is not because SQL was wrong — it's because doing analysis 
in a second tool forces more careful computation and reading.

## Electronics Divergence — the strongest intra-sector story

Two companies. Same sector label. Opposite outcomes over 13 years.

| Metric                            | INTC      | NVDA      | Ratio                     |
|---                                |---        |---        |---                        |
| 2009 Revenue                      | $35B      | $3.5B     | INTC 10× larger           |
| 2022 Revenue                      | $63B      | $27B      | INTC 2.3× larger          |
| Revenue CAGR                      | 4.60%     | 17.18%    | NVDA 3.7× faster          |
| 2009 Market Cap                   | $112.7B   | $10.4B    | INTC 11× larger           |
| 2022 Market Cap                   | $109.1B   | $359.5B   | **NVDA 3.3× larger**      |
| Market Cap multiple (2022/2009)   | 1.0×      | 34.7×     | **NVDA grew 35.8× more**  |

**The valuation reversal is striking.** INTC entered the period 11× larger 
by market cap. By 2022, NVDA had overtaken INTC by 3.3×. The reversal came 
from NVDA's positioning for the GPU/AI/data-centre wave while INTC 
remained CPU-focused.

**The 2017 inflection.** NVDA revenue grew from $5B (2016) to $27B (2022) 
— roughly 40% CAGR over the last six years of the analysis window. This 
reflects the emergence of data-centre GPU demand, cryptocurrency mining 
cycles, and early AI/ML infrastructure investment. Notably, our dataset 
ends before the 2023 generative-AI surge that pushed NVDA's market cap 
above $2T in 2024.

**Sector-level Electronics CAGR of 6.7% is analytically almost meaningless.** 
It's the average of stagnation (INTC) and hyper-growth (NVDA). An analyst 
reporting only the sector figure would miss:
- One company's positioning captured a paradigm shift  
- The other missed the same shift
- The dispersion within the sector was greater than any typical between-sector 
  comparison

**Broader principle:** Sector labels group companies by *industry classification*, 
not by *strategic positioning*. When sector composition contains companies 
at different points of technological transition, sector-level aggregates 
mask the real analytical story.

In [7]:
# Year on Year Growth Comparison
electronics_df["revenue_yoy_pct"] = (
    electronics_df.groupby("Company")["Revenue"].pct_change() * 100
).round(1)

electronics_yoy = electronics_df.pivot(
    index="Year",
    columns="Company",
    values="revenue_yoy_pct"
)
electronics_yoy

Company,INTC,NVDA
Year,,
2009,NaN,NaN
2010,24.2,-2.9
2011,23.8,6.5
2012,-1.2,12.8
2013,-1.2,7.1
2014,6.0,-3.5
2015,-0.9,13.4
2016,7.3,7.0
2017,5.7,37.9


In [6]:
# Growth Multiples - How many times each company's market cap grew from 2009 to 2022.
# INTC multiple
intc_mcap = electronics_df[electronics_df["Company"] == "INTC"].sort_values("Year")
intc_start = intc_mcap.iloc[0]["Market Cap(in B USD)"]
intc_end = intc_mcap.iloc[-1]["Market Cap(in B USD)"]
intc_multiple = intc_end / intc_start

# NVDA multiple
nvda_mcap = electronics_df[electronics_df["Company"] == "NVDA"].sort_values("Year")
nvda_start = nvda_mcap.iloc[0]["Market Cap(in B USD)"]
nvda_end = nvda_mcap.iloc[-1]["Market Cap(in B USD)"]
nvda_multiple = nvda_end / nvda_start

print(f"INTC market cap 2009: ${intc_start:.1f}B -> 2022: ${intc_end:.1f}B ({intc_multiple:.1f}× multiple)")
print(f"NVDA market cap 2009: ${nvda_start:.1f}B -> 2022: ${nvda_end:.1f}B ({nvda_multiple:.1f}× multiple)")
print()
print(f"Ratio of NVDA growth to INTC growth: {nvda_multiple / intc_multiple:.1f}×")

INTC market cap 2009: $112.7B -> 2022: $109.1B (1.0× multiple)
NVDA market cap 2009: $10.4B -> 2022: $359.5B (34.7× multiple)

Ratio of NVDA growth to INTC growth: 35.8×


In [5]:
# Market Cap Comparison
electronics_mcap = electronics_df.pivot(
    index="Year",
    columns="Company",
    values="Market Cap(in B USD)"
).round(1)
electronics_mcap

Company,INTC,NVDA
Year,,
2009,112.6,10.4
2010,117.3,9.0
2011,123.5,8.5
2012,102.6,7.7
2013,129.0,9.1
2014,175.5,10.9
2015,162.6,17.7
2016,171.9,57.5
2017,216.0,117.3


In [4]:
# CAGR for Both Companies (INTC & NVDA)
def compute_company_cagr(group):
    first_year_row = group.loc[group["Year"].idxmin()]
    last_year_row = group.loc[group["Year"].idxmax()]
    years = last_year_row["Year"] - first_year_row["Year"]
    if years == 0 or first_year_row["Revenue"] <= 0:
        return None
    return ((last_year_row["Revenue"] / first_year_row["Revenue"]) ** (1 / years) - 1) * 100

electronics_cagr = electronics_df.groupby("Company").apply(
    compute_company_cagr, include_groups=False
).round(2)
electronics_cagr

Company
INTC     4.60
NVDA    17.18
dtype: float64

In [3]:
# Revenue Side by Side
electronics_revenue = electronics_df.pivot(
    index="Year",
    columns="Company",
    values="Revenue"
).round(0)
electronics_revenue

Company,INTC,NVDA
Year,,
2009,35127.0,3425.0
2010,43623.0,3326.0
2011,53999.0,3543.0
2012,53341.0,3998.0
2013,52708.0,4280.0
2014,55870.0,4130.0
2015,55355.0,4682.0
2016,59387.0,5010.0
2017,62761.0,6910.0


In [2]:
# Electronics Divergence
# The question: INTC and NVDA share the "Electronics" sector label. Do their trajectories 2009-2022 support the "sector matters" narrative, or is it "positioning within a sector matters more"?
# Approach: compare them directly on revenue, market cap, and growth rate.

# Filter & Inspect Electronics
electronics_df = df[df["sector"] == "Electronics"].copy()

# Sort explicitly (learned lesson from Block 2)
electronics_df = electronics_df.sort_values(["Company", "Year"])

print(f"Electronics rows: {len(electronics_df)}")
print(f"Companies: {electronics_df['Company'].unique()}")
print(f"Year range: {electronics_df['Year'].min()} to {electronics_df['Year'].max()}")

Electronics rows: 28
Companies: <StringArray>
['INTC', 'NVDA']
Length: 2, dtype: str
Year range: 2009 to 2022


In [1]:
import pandas as pd
from pathlib import Path

SCRIPT_DIR = Path().resolve()
df = pd.read_csv(SCRIPT_DIR / "data" / "Financial Statements.csv")
df.columns = df.columns.str.strip()

sector_map = {
    "IT": "Technology", "LOGI": "Logistics", "FOOD": "Food & Beverage",
    "BANK": "Banking", "ELEC": "Electronics", "FinTech": "FinTech",
    "Finance": "Finance", "Manufacturing": "Manufacturing",
}
df["sector"] = df["Category"].map(sector_map)
df = df[(df["Year"] >= 2009) & (df["Year"] <= 2022)]

print(df.shape)

(159, 24)
